# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


## Objective

This notebook implements the **BM25 (Best Matching 25)** sparse retrieval model as the first baseline for evaluating the proposed **Root-Aware Lexical Semantic Retrieval (RALSR)** framework.

BM25 is one of the most widely adopted lexical retrieval algorithms in Information Retrieval (IR). Unlike the proposed RALSR framework, BM25 relies exclusively on lexical keyword matching without incorporating Arabic morphological analysis, hybrid root extraction, semantic enrichment, or knowledge-driven ranking. Documents are ranked according to the BM25 relevance function, which combines term frequency, inverse document frequency, and document length normalization.

The primary objective of this notebook is to establish a strong classical retrieval baseline that will later be compared with:

- **Baseline 1:** BM25 Sparse Retrieval (this notebook)
- **Baseline 2:** Dense Retrieval using Pre-trained Embeddings (Notebook 10B)
- **Proposed Method A:** Root-Aware Lexical Semantic Retrieval (RALSR)
- **Proposed Method B:** RALSR + Transformer Cross-Encoder Re-ranking (Notebook 11)

The corrected run is generated for all 251 authoritative questions independently of qrels and is preserved for later controlled evaluation.

---

# Workflow

```text
Preprocessed Questions
           │
           ▼
Processed Qur'anic Passage Collection
           │
           ▼
Tokenized Passage Corpus
           │
           ▼
BM25 Index Construction
           │
           ▼
BM25 Passage Scoring
           │
           ▼
Passage Ranking
           │
           ▼
Export Ranked Retrieval Results
           │
           ▼
Later controlled evaluation
```

---

# Input Datasets

## 1. Preprocessed Query Collection

Generated and validated in the corrected **Notebook 4 – Arabic Query Preprocessing** stage

**File**

`QuranQA_Preprocessed_Queries_Corrected.csv`

Contains:

- Question_ID
- Question
- Query_Terms

---

## 2. Processed Qur'anic Passage Collection

Generated in **Notebook 8 – Preparing the Quran Passage Collection for Retrieval**

**File**

`QuranQA_Processed_Passage_Collection.csv`

Contains:

- Passage_ID
- Passage_Text
- Tokens

---

# BM25 Retrieval Pipeline

The BM25 baseline follows the standard sparse retrieval paradigm:

1. Load the preprocessed questions.
2. Load the processed Qur'anic passage collection.
3. Construct the BM25 index using the tokenized passage corpus.
4. Compute BM25 relevance scores for every question–passage pair.
5. Rank passages according to their BM25 scores.
6. Export the ranked retrieval results.

Unlike the proposed RALSR framework, BM25 does **not** perform:

- Hybrid root extraction
- Canonical lexical mapping (CLM)
- Semantic enrichment using *Maqāyīs al-Lughah*
- Linguistic Evidence Model (LEM) construction
- Knowledge-driven feature weighting

Instead, retrieval is performed exclusively through lexical keyword matching.

---

# Output

This notebook produces:

`BM25_Retrieval_Top100_Corrected.csv`

The exported dataset contains:

- Question_ID
- Question
- Passage_ID
- BM25_Score
- Rank

The resulting ranked retrieval list serves as the **classical lexical retrieval baseline** for later controlled evaluation.

In [1]:
# ============================================================
# Import Required Libraries
# Notebook 10A – Baseline 1: BM25 Sparse Retrieval
# ============================================================

import ast
from pathlib import Path

import numpy as np
import pandas as pd


def find_workspace_root(start: Path) -> Path:
    """Locate the thesis writing workspace from the execution directory."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "00_Project_Control").is_dir()
            and (candidate / "02_Methodology_Evidence").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the thesis workspace. Run this notebook from within "
        "the Master Thesis - Quranic Passage Retrieval - Writing Workspace tree."
    )


PROJECT_ROOT = find_workspace_root(Path.cwd())
METHODOLOGY_DIR = PROJECT_ROOT / "02_Methodology_Evidence"

QUERY_FILE = (
    METHODOLOGY_DIR
    / "QAC and Root Extraction"
    / "Corrected_Outputs"
    / "Point_2_Query_Root_Provenance"
    / "QuranQA_Preprocessed_Queries_Corrected.csv"
)

PASSAGE_FILE = (
    METHODOLOGY_DIR
    / "Data and Resources"
    / "Corrected_Outputs"
    / "QPC_Passage_Side_Validation"
    / "QuranQA_Processed_Passage_Collection.csv"
)

BM25_OUTPUT_DIR = (
    METHODOLOGY_DIR
    / "Baselines"
    / "Corrected_Outputs"
    / "Base_BM25_Corrected_Query"
)
BM25_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("PROJECT PATHS INITIALIZED")
print("=" * 70)
print(f"Project Root        : {PROJECT_ROOT}")
print(f"Corrected Query File: {QUERY_FILE}")
print(f"Processed QPC File  : {PASSAGE_FILE}")
print(f"Output Folder       : {BM25_OUTPUT_DIR}")


PROJECT PATHS INITIALIZED
Project Root        : ${THESIS_WORKSPACE}
Corrected Query File: ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance\QuranQA_Preprocessed_Queries_Corrected.csv
Processed QPC File  : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Data and Resources\Corrected_Outputs\QPC_Passage_Side_Validation\QuranQA_Processed_Passage_Collection.csv
Output Folder       : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Baselines\Corrected_Outputs\Base_BM25_Corrected_Query


In [2]:
# ============================================================
# Load and Validate Authoritative BM25 Resources
# ============================================================

queries_df = pd.read_csv(QUERY_FILE, encoding="utf-8-sig")
passages_df = pd.read_csv(PASSAGE_FILE, encoding="utf-8-sig")

queries_df["Query_Terms"] = queries_df["Query_Terms"].apply(ast.literal_eval)
passages_df["Tokens"] = passages_df["Passage_Text"].fillna("").str.split()

selected_term_count = int(queries_df["Query_Terms"].map(len).sum())

if len(queries_df) != 251 or queries_df["Question_ID"].nunique() != 251:
    raise ValueError("Corrected query input must contain 251 unique question IDs.")
if selected_term_count != 1342:
    raise ValueError(f"Expected 1,342 corrected query terms, found {selected_term_count:,}.")
if 504 not in set(pd.to_numeric(queries_df["Question_ID"], errors="raise").astype(int)):
    raise ValueError("qid 504 is missing from the corrected query input.")
if len(passages_df) != 1266 or passages_df["Passage_ID"].nunique() != 1266:
    raise ValueError("Processed QPC must contain 1,266 unique passage IDs.")
if passages_df["Passage_ID"].isna().any() or passages_df["Passage_Text"].isna().any():
    raise ValueError("Processed QPC contains a blank passage ID or passage text.")

print("=" * 70)
print("AUTHORITATIVE BM25 RESOURCES LOADED")
print("=" * 70)
print(f"Questions             : {len(queries_df):,}")
print(f"Selected Query Terms  : {selected_term_count:,}")
print(f"Passages              : {len(passages_df):,}")
print(f"Unique Passage IDs    : {passages_df['Passage_ID'].nunique():,}")
print("Qrel data loaded      : No")

print("\nSample Query")
display(queries_df[["Question_ID", "Question", "Query_Terms"]].head(1))

print("\nSample Passage")
display(passages_df[["Passage_ID", "Passage_Text", "Tokens"]].head(1))


AUTHORITATIVE BM25 RESOURCES LOADED
Questions             : 251
Selected Query Terms  : 1,342
Passages              : 1,266
Unique Passage IDs    : 1,266
Qrel data loaded      : No

Sample Query


,Question_ID,Question,Query_Terms
0,101,من هم قوم شعيب؟,"[قوم, شعيب]"



Sample Passage


,Passage_ID,Passage_Text,Tokens
0,1:1-4,بسم الله الرحمن الرحيم. الحمد لله رب العالمين....,"[بسم, الله, الرحمن, الرحيم., الحمد, لله, رب, ا..."


In [3]:
# ============================================================
# Build BM25 Corpus and Index
# ============================================================

from rank_bm25 import BM25Okapi

BM25_K1 = 1.5
BM25_B = 0.75
BM25_EPSILON = 0.25

bm25_corpus = passages_df["Tokens"].tolist()
bm25_index = BM25Okapi(
    bm25_corpus,
    k1=BM25_K1,
    b=BM25_B,
    epsilon=BM25_EPSILON,
)

document_lengths = [len(document) for document in bm25_corpus]

print("=" * 70)
print("BM25 INDEX CREATED")
print("=" * 70)
print(f"Indexed Passages      : {len(bm25_corpus):,}")
print(f"k1 / b / epsilon      : {BM25_K1} / {BM25_B} / {BM25_EPSILON}")
print(f"Average Document Size : {np.mean(document_lengths):.2f} tokens")
print(f"Minimum Length        : {np.min(document_lengths)} tokens")
print(f"Maximum Length        : {np.max(document_lengths)} tokens")
print("\nSample Tokenized Passage\n")
print(bm25_corpus[0][:30])


BM25 INDEX CREATED
Indexed Passages      : 1,266
k1 / b / epsilon      : 1.5 / 0.75 / 0.25
Average Document Size : 61.54 tokens
Minimum Length        : 7 tokens
Maximum Length        : 219 tokens

Sample Tokenized Passage

['بسم', 'الله', 'الرحمن', 'الرحيم.', 'الحمد', 'لله', 'رب', 'العالمين.', 'الرحمن', 'الرحيم.', 'مالك', 'يوم', 'الدين.']


## BM25 Passage Retrieval

The BM25 retrieval model is applied to the entire Qur'anic passage collection to estimate the lexical relevance of each passage with respect to every preprocessed query.

For each question:

1. The preprocessed query terms are submitted to the BM25 retrieval model.
2. BM25 relevance scores are computed for all indexed passages.
3. One retrieval score is generated for every question–passage pair.
4. The resulting retrieval scores are stored for subsequent ranking.

Unlike the proposed **Root-Aware Lexical Semantic Retrieval (RALSR)** framework, which first performs candidate passage retrieval before knowledge-driven ranking, BM25 evaluates the complete passage collection directly using lexical term matching.

The resulting retrieval scores constitute the classical sparse retrieval baseline used for comparison with the proposed framework.

In [4]:
# ============================================================
# BM25 Passage Retrieval and Scoring
# ============================================================

bm25_results = []

# ------------------------------------------------------------
# Score Every Question Against Every Passage
# ------------------------------------------------------------

for _, query_row in queries_df.iterrows():

    query_terms = query_row["Query_Terms"]

    # Compute BM25 scores for the entire corpus
    scores = bm25_index.get_scores(query_terms)

    # Store one record per passage
    for passage_row, score in zip(
        passages_df.itertuples(index=False),
        scores
    ):

        bm25_results.append({

            "Question_ID": query_row["Question_ID"],
            "Question": query_row["Question"],
            "Passage_ID": passage_row.Passage_ID,
            "BM25_Score": float(score)

        })

# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

bm25_df = pd.DataFrame(bm25_results)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("=" * 70)
print("BM25 RETRIEVAL COMPLETED")
print("=" * 70)

print(f"Question–Passage Pairs : {len(bm25_df):,}")
print(f"Questions Processed    : {queries_df['Question_ID'].nunique():,}")
print(f"Indexed Passages       : {passages_df['Passage_ID'].nunique():,}")

print(f"\nMinimum Score : {bm25_df['BM25_Score'].min():.4f}")
print(f"Maximum Score : {bm25_df['BM25_Score'].max():.4f}")
print(f"Average Score : {bm25_df['BM25_Score'].mean():.4f}")

print("\nSample Retrieval Scores")

display(
    bm25_df.head(10)
)

BM25 RETRIEVAL COMPLETED
Question–Passage Pairs : 317,766
Questions Processed    : 251
Indexed Passages       : 1,266

Minimum Score : 0.0000
Maximum Score : 58.3699
Average Score : 0.3837

Sample Retrieval Scores


,Question_ID,Question,Passage_ID,BM25_Score
0,101,من هم قوم شعيب؟,1:1-4,0.0
1,101,من هم قوم شعيب؟,1:5-6,0.0
2,101,من هم قوم شعيب؟,1:7-7,0.0
3,101,من هم قوم شعيب؟,2:1-2,0.0
4,101,من هم قوم شعيب؟,2:3-5,0.0
5,101,من هم قوم شعيب؟,2:6-7,0.0
6,101,من هم قوم شعيب؟,2:8-16,0.0
7,101,من هم قوم شعيب؟,2:17-20,0.0
8,101,من هم قوم شعيب؟,2:21-22,0.0
9,101,من هم قوم شعيب؟,2:23-24,0.0


## BM25 Passage Ranking

After computing the BM25 relevance scores for every question–passage pair, the retrieved passages are ranked independently for each question.

Passages are sorted in descending order according to their BM25 scores, where the highest-scoring passage receives **Rank 1**. The resulting ranked lists represent the final retrieval output of the BM25 baseline and provide a direct comparison with the ranked outputs produced by the proposed Root-Aware Lexical Semantic Retrieval (RALSR) framework.

The ranked retrieval results generated in this section will be exported for later controlled evaluation.

In [5]:
# ============================================================
# Rank BM25 Retrieval Results Deterministically
# ============================================================

# Descending BM25 score; equal scores use ascending official Passage_ID.
bm25_df = bm25_df.sort_values(
    by=["Question_ID", "BM25_Score", "Passage_ID"],
    ascending=[True, False, True],
    kind="mergesort",
).reset_index(drop=True)

bm25_df["Rank"] = bm25_df.groupby("Question_ID").cumcount() + 1

print("=" * 70)
print("BM25 RANKING COMPLETED")
print("=" * 70)
print(f"Total Ranked Pairs : {len(bm25_df):,}")
print(f"Questions Ranked   : {bm25_df['Question_ID'].nunique():,}")
print("Tie Ordering       : Ascending Passage_ID")

print("\nTop-ranked passage for the first five questions\n")
display(bm25_df.groupby("Question_ID").head(1).head(5))

print("\nRank Distribution\n")
display(bm25_df["Rank"].describe())


BM25 RANKING COMPLETED
Total Ranked Pairs : 317,766


Questions Ranked   : 251
Tie Ordering       : Ascending Passage_ID

Top-ranked passage for the first five questions



,Question_ID,Question,Passage_ID,BM25_Score,Rank
0,101,من هم قوم شعيب؟,11:89-95,8.855709,1
1266,102,من هم قوم موسى؟,11:89-95,4.803170,1
2532,103,من بنى الكعبة؟,5:94-95,5.668227,1
3798,105,من هو النبي المعروف بالصبر؟,9:67-68,8.360193,1
5064,106,من كفل السيدة مريم؟,3:42-44,7.139392,1



Rank Distribution



count    317766.000000
mean        633.500000
std         365.463181
min           1.000000
25%         317.000000
50%         633.500000
75%         950.000000
max        1266.000000
Name: Rank, dtype: float64

## Top-K BM25 Retrieval

Although BM25 computes a relevance score for every passage in the collection, Information Retrieval systems typically return only the highest-ranked documents to the user.

In this study, the **Top-100** ranked passages are retained for each question. Restricting the retrieval results to the highest-scoring candidates reduces the size of the retrieval output while preserving all passages required for subsequent evaluation.

The resulting Top-100 ranked retrieval lists constitute the final BM25 retrieval output used for later controlled evaluation.

In [6]:
# ============================================================
# Retrieve Top-100 BM25 Passages
# ============================================================

TOP_K = 100

# ------------------------------------------------------------
# Keep Only Top-K Ranked Passages per Question
# ------------------------------------------------------------

bm25_topk_df = (
    bm25_df
    .groupby("Question_ID", group_keys=False)
    .head(TOP_K)
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("=" * 70)
print("TOP-100 BM25 RETRIEVAL COMPLETED")
print("=" * 70)

print(f"Questions Retrieved : {bm25_topk_df['Question_ID'].nunique():,}")
print(f"Top-K               : {TOP_K}")
print(f"Retrieved Records   : {len(bm25_topk_df):,}")

print("\nSample Top-10 Retrieved Passages")

display(
    bm25_topk_df.head(10)
)

TOP-100 BM25 RETRIEVAL COMPLETED
Questions Retrieved : 251
Top-K               : 100
Retrieved Records   : 25,100

Sample Top-10 Retrieved Passages


,Question_ID,Question,Passage_ID,BM25_Score,Rank
0,101,من هم قوم شعيب؟,11:89-95,8.855709,1
1,101,من هم قوم شعيب؟,11:84-88,7.784184,2
2,101,من هم قوم شعيب؟,7:85-93,5.718353,3
3,101,من هم قوم شعيب؟,26:176-191,4.349822,4
4,101,من هم قوم شعيب؟,7:138-140,4.114427,5
5,101,من هم قوم شعيب؟,40:28-33,4.027243,6
6,101,من هم قوم شعيب؟,6:132-135,3.889493,7
7,101,من هم قوم شعيب؟,21:74-75,3.659176,8
8,101,من هم قوم شعيب؟,21:76-77,3.585414,9
9,101,من هم قوم شعيب؟,29:36-37,3.585414,10


## Export BM25 Retrieval Results

The BM25 retrieval results are exported in two formats:

1. **Complete Ranked Retrieval Results** – Contains the ranking of all query–passage pairs for reproducibility and future analyses.

2. **Top-100 Retrieval Results** – Contains the highest-ranked 100 passages for each question and serves as the primary input for the later controlled evaluation.

Both corrected files are stored in the isolated base-BM25 corrected-output directory; the historical output remains unchanged.

In [7]:
# ============================================================
# Export Corrected BM25 Retrieval Results
# ============================================================

ALL_RESULTS_FILE = BM25_OUTPUT_DIR / "BM25_Retrieval_All_Ranked_Corrected.csv"
TOP100_RESULTS_FILE = BM25_OUTPUT_DIR / "BM25_Retrieval_Top100_Corrected.csv"

query_lookup = queries_df[["Question_ID", "Query_Terms"]]
bm25_df = bm25_df.merge(query_lookup, on="Question_ID", how="left")
bm25_topk_df = bm25_topk_df.merge(query_lookup, on="Question_ID", how="left")

column_order = [
    "Question_ID",
    "Question",
    "Query_Terms",
    "Passage_ID",
    "BM25_Score",
    "Rank",
]
bm25_df = bm25_df[column_order]
bm25_topk_df = bm25_topk_df[column_order]

bm25_df.to_csv(ALL_RESULTS_FILE, index=False, encoding="utf-8-sig")
bm25_topk_df.to_csv(TOP100_RESULTS_FILE, index=False, encoding="utf-8-sig")

print("=" * 70)
print("CORRECTED BM25 RETRIEVAL RESULTS EXPORTED")
print("=" * 70)
print(f"Complete Ranked Results : {ALL_RESULTS_FILE.name}")
print(f"Top-100 Results         : {TOP100_RESULTS_FILE.name}")
print(f"Complete Ranked Rows    : {len(bm25_df):,}")
print(f"Top-100 Rows            : {len(bm25_topk_df):,}")
print(f"Questions               : {bm25_topk_df['Question_ID'].nunique():,}")
print(f"Top-K                   : {TOP_K}")
print(f"Files Saved To          : {BM25_OUTPUT_DIR}")


CORRECTED BM25 RETRIEVAL RESULTS EXPORTED
Complete Ranked Results : BM25_Retrieval_All_Ranked_Corrected.csv
Top-100 Results         : BM25_Retrieval_Top100_Corrected.csv
Complete Ranked Rows    : 317,766
Top-100 Rows            : 25,100
Questions               : 251
Top-K                   : 100
Files Saved To          : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Baselines\Corrected_Outputs\Base_BM25_Corrected_Query


In [8]:
# ============================================================
# Display Top-10 BM25 Retrieval Results
# ============================================================

SAMPLE_QUESTION_ID = 101

sample_results = (
    bm25_topk_df[
        bm25_topk_df["Question_ID"] == SAMPLE_QUESTION_ID
    ]
    .head(10)
    .copy()
)

print("=" * 70)
print("TOP 10 BM25 RETRIEVAL RESULTS")
print("=" * 70)

print(f"Question ID : {SAMPLE_QUESTION_ID}")

print(
    f"Question    : "
    f"{sample_results.iloc[0]['Question']}"
)

print("\nTop-10 Retrieved Passages\n")

display(
    sample_results[
        [
            "Rank",
            "Passage_ID",
            "BM25_Score"
        ]
    ]
)

TOP 10 BM25 RETRIEVAL RESULTS
Question ID : 101
Question    : من هم قوم شعيب؟

Top-10 Retrieved Passages



,Rank,Passage_ID,BM25_Score
0,1,11:89-95,8.855709
1,2,11:84-88,7.784184
2,3,7:85-93,5.718353
3,4,26:176-191,4.349822
4,5,7:138-140,4.114427
5,6,40:28-33,4.027243
6,7,6:132-135,3.889493
7,8,21:74-75,3.659176
8,9,21:76-77,3.585414
9,10,29:36-37,3.585414


## Qualitative Inspection of BM25 Retrieval Results

To illustrate the behavior of the BM25 baseline, the highest-ranked passages retrieved for a representative question are examined qualitatively.

The retrieved passages are presented together with their BM25 relevance scores and the corresponding Qur'anic passage text. This qualitative inspection complements the quantitative evaluation by demonstrating the types of passages favored by lexical retrieval.

A similar qualitative inspection will be performed for the Dense Retrieval baseline and the proposed Root-Aware Lexical Semantic Retrieval (RALSR) framework to facilitate direct comparison of the retrieved passages.

In [9]:
# ============================================================
# Qualitative Inspection of BM25 Retrieval Results
# ============================================================

SAMPLE_QUESTION_ID = 101

# ------------------------------------------------------------
# Retrieve Top-10 Results
# ------------------------------------------------------------

sample_results = (
    bm25_topk_df[
        bm25_topk_df["Question_ID"] == SAMPLE_QUESTION_ID
    ]
    .head(10)
    .copy()
)

# ------------------------------------------------------------
# Attach Passage Text
# ------------------------------------------------------------

sample_results = sample_results.merge(
    passages_df[
        [
            "Passage_ID",
            "Passage_Text"
        ]
    ],
    on="Passage_ID",
    how="left"
)

# ------------------------------------------------------------
# Display Results
# ------------------------------------------------------------

print("=" * 70)
print("TOP 10 BM25 RETRIEVAL RESULTS")
print("=" * 70)

print(f"Question ID : {SAMPLE_QUESTION_ID}")
print(f"Question    : {sample_results.iloc[0]['Question']}")
print(f"Query Terms : {sample_results.iloc[0]['Query_Terms']}")

print("\nTop-10 Retrieved Passages\n")

display(
    sample_results[
        [
            "Rank",
            "Passage_ID",
            "BM25_Score",
            "Passage_Text"
        ]
    ]
)

TOP 10 BM25 RETRIEVAL RESULTS
Question ID : 101
Question    : من هم قوم شعيب؟
Query Terms : ['قوم', 'شعيب']

Top-10 Retrieved Passages



,Rank,Passage_ID,BM25_Score,Passage_Text
0,1,11:89-95,8.855709,ويا قوم لا يجرمنكم شقاقي أن يصيبكم مثل ما أصاب...
1,2,11:84-88,7.784184,وإلى مدين أخاهم شعيبا قال يا قوم اعبدوا الله م...
2,3,7:85-93,5.718353,وإلى مدين أخاهم شعيبا قال يا قوم اعبدوا الله م...
3,4,26:176-191,4.349822,كذب أصحاب الأيكة المرسلين. إذ قال لهم شعيب ألا...
4,5,7:138-140,4.114427,وجاوزنا ببني إسرائيل البحر فأتوا على قوم يعكفو...
5,6,40:28-33,4.027243,وقال رجل مؤمن من آل فرعون يكتم إيمانه أتقتلون ...
6,7,6:132-135,3.889493,ولكل درجات مما عملوا وما ربك بغافل عما يعملون....
7,8,21:74-75,3.659176,ولوطا آتيناه حكما وعلما ونجيناه من القرية التي...
8,9,21:76-77,3.585414,ونوحا إذ نادى من قبل فاستجبنا له فنجيناه وأهله...
9,10,29:36-37,3.585414,وإلى مدين أخاهم شعيبا فقال يا قوم اعبدوا الله ...


# Notebook Summary

## Overview

This notebook implemented the **BM25 (Best Matching 25)** sparse retrieval model as the first baseline for evaluating the proposed **Root-Aware Lexical Semantic Retrieval (RALSR)** framework.

Using the final corrected Notebook-4 QuranQA queries and the processed Qur'anic passage collection, a BM25 retrieval index was constructed to compute lexical relevance scores between each question and every passage in the collection. The retrieved passages were subsequently ranked according to their BM25 scores, and the Top-100 passages for each question were retained as the final retrieval output.

---

## Accomplishments

The following tasks were successfully completed:

- Loaded the preprocessed QuranQA query collection.
- Loaded the processed Qur'anic passage collection.
- Constructed the BM25 retrieval index.
- Computed BM25 relevance scores for every corrected-query–passage pair.
- Ranked all passages according to their BM25 scores.
- Retrieved the Top-100 ranked passages for all 251 authoritative questions independently of qrels.
- Exported the final BM25 retrieval results for subsequent evaluation.
- Performed a qualitative inspection of the retrieved passages.

---

## Output

The notebook generated the following dataset:

**`BM25_Retrieval_Top100_Corrected.csv`**

This dataset contains the Top-100 ranked BM25 retrieval results for all 251 authoritative questions independently of qrels and is preserved as the corrected classical lexical baseline for later controlled evaluation.

---

## Next Step

The corrected base-BM25 run is preserved for later controlled evaluation. No effectiveness metric is calculated in this notebook.

---

## Corrected-Query Run Validation

- Authoritative questions: **251**
- Corrected Notebook-4 selected terms: **1,342** (historical: 1,435)
- Processed QPC passages: **1,266**, byte-identical to the validated Notebook-8 output
- Corrected Top-100 rows: **25,100**
- qid 504: **included**
- Retrieval generation uses qrels: **No**
- Invalid QPC IDs, duplicate pairs, non-finite scores, rank-continuity errors, and score-order errors: **0**
- Determinism: **passed across two complete BM25 executions**
- Final Top-100 SHA-256: `B24A5946FF821136352616B65B6C9AA1C1D66279114F8780F95D8425E1803235`

No effectiveness metric was calculated in this notebook.
